# Análise por Carta: Volume de Corais e Solos vs Pontuação (por AGENTE)

Para cada jogo, o `results.csv` guarda quantos tiles de cada solo e quantos corais de cada tipo cada
jogador colocou, junto do score e de **qual agente** ocupou cada cadeira (`p1_agent`/`p2_agent`).
Aqui cruzamos **volume por tipo** com a **pontuação**, rotulando por agente.

> Se o torneio usou `randomize_seats=True`, cada agente jogou metade em cada cadeira, então as médias
> por agente já são justas (sem viés de posição). A regressão de "pontos marginais por carta" independe
> do agente — mede o valor da carta em si.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS_PATH = None


def find_latest_results_csv(root: Path) -> Path:
    base = root / 'artifacts' / 'tournaments'
    dirs = sorted(base.glob('*/'), key=lambda p: p.stat().st_mtime, reverse=True)
    for d in dirs:
        candidate = d / 'results.csv'
        if candidate.exists() and any(c.startswith('p1_coral_') for c in pd.read_csv(candidate, nrows=0).columns):
            return candidate
    raise FileNotFoundError('Nenhum results.csv com colunas de volume por tipo (p1_coral_*). Rode um torneio.')


results_path = Path(RESULTS_PATH) if RESULTS_PATH else find_latest_results_csv(ROOT)
print('Usando:', results_path)
df = pd.read_csv(results_path)


def read_meta(csv_path):
    meta_path = csv_path.parent / 'metadata.json'
    if meta_path.exists():
        return json.loads(meta_path.read_text(encoding='utf-8')).get('tournament_config', {})
    return {}

meta = read_meta(results_path)
# Garante p1_agent/p2_agent (novos artefatos já têm; antigos derivam do metadata).
if 'p1_agent' not in df.columns:
    df['p1_agent'] = meta.get('agent_p1', 'P1')
    df['p2_agent'] = meta.get('agent_p2', 'P2')

print('jogos:', len(df), '| P1 =', meta.get('agent_p1'), '| P2 =', meta.get('agent_p2'),
      '| randomize_seats =', meta.get('randomize_seats'))

## Reformatar para "long" por AGENTE

In [ ]:
coral_types = sorted(c[len('p1_coral_'):] for c in df.columns if c.startswith('p1_coral_'))
soil_types = sorted(c[len('p1_soil_'):] for c in df.columns if c.startswith('p1_soil_'))
fauna_types = sorted(
    c[len('p1_fauna_'):] for c in df.columns
    if c.startswith('p1_fauna_') and not c.endswith('_total')
)
print('corais:', coral_types)
print('solos:', soil_types)
print('fauna:', fauna_types)

frames = []
for pid in (1, 2):
    sub = pd.DataFrame({
        'seed': df['seed'],
        'agent': df[f'p{pid}_agent'],
        'score': df[f'p{pid}_score'],
    })
    for ct in coral_types:
        sub[f'coral_{ct}'] = df[f'p{pid}_coral_{ct}']
    for st in soil_types:
        sub[f'soil_{st}'] = df[f'p{pid}_soil_{st}']
    for ft in fauna_types:
        sub[f'fauna_{ft}'] = df[f'p{pid}_fauna_{ft}']
    frames.append(sub)

long = pd.concat(frames, ignore_index=True)
vol_cols = (
    [f'coral_{c}' for c in coral_types]
    + [f'soil_{s}' for s in soil_types]
    + [f'fauna_{f}' for f in fauna_types]
)
long.head()

## Volume médio e score por AGENTE

In [ ]:
mean_by_agent = long.groupby('agent')[['score'] + vol_cols].mean().round(2)
mean_by_agent

## Correlação de cada volume com o score (agregado)

In [ ]:
corr = long[vol_cols + ['score']].corr()['score'].drop('score').sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(corr.index, corr.values, color=['tab:green' if v >= 0 else 'tab:red' for v in corr.values])
ax.axvline(0, color='k', linewidth=0.8)
ax.set_title('Correlação (Pearson) do volume de cada carta com o score')
ax.set_xlabel('correlação com score')
plt.tight_layout(); plt.show()

## Dispersão: volume vs score (por agente)

In [ ]:
agents = sorted(long['agent'].unique())
palette = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple']
agent_color = {a: palette[i % len(palette)] for i, a in enumerate(agents)}


def scatter_grid(cols, title):
    n = len(cols)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4), squeeze=False)
    for ax, col in zip(axes[0], cols):
        for a in agents:
            sub = long[long.agent == a]
            ax.scatter(sub[col], sub['score'], s=12, alpha=0.35, color=agent_color[a], label=a)
        trend = long.groupby(col)['score'].mean()
        ax.plot(trend.index, trend.values, color='k', marker='o', markersize=3, label='média')
        ax.set_title(col); ax.set_xlabel('volume'); ax.set_ylabel('score')
    axes[0][0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()


scatter_grid([f'coral_{c}' for c in coral_types], 'Corais: volume vs score')
scatter_grid([f'soil_{s}' for s in soil_types], 'Solos: volume vs score')
if fauna_types:
    scatter_grid([f'fauna_{f}' for f in fauna_types], 'Fauna: volume vs score')

## Pontos marginais por carta (regressão múltipla)

`score ~ intercepto + Σ volumes`. Mede o valor da carta em si (independe do agente).

In [ ]:
X = np.column_stack([np.ones(len(long))] + [long[c].values.astype(float) for c in vol_cols])
y = long['score'].values.astype(float)
coef, *_ = np.linalg.lstsq(X, y, rcond=None)
reg = pd.Series(coef, index=['(intercepto)'] + vol_cols).round(2).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
body = reg.drop('(intercepto)')
ax.barh(body.index, body.values, color=['tab:green' if v >= 0 else 'tab:red' for v in body.values])
ax.axvline(0, color='k', linewidth=0.8)
ax.set_title('Pontos marginais por carta (coeficiente da regressão)')
ax.set_xlabel('pontos por unidade')
plt.tight_layout(); plt.show()
reg.to_frame('pontos_por_carta')

## Tabela-resumo por tipo de carta

In [ ]:
summary = pd.DataFrame({
    'volume_medio': long[vol_cols].mean().round(2),
    'volume_max': long[vol_cols].max(),
    'corr_com_score': long[vol_cols].corrwith(long['score']).round(3),
    'pontos_marginais': reg.reindex(vol_cols).round(2),
})
summary.sort_values('pontos_marginais', ascending=False)